# Hackathon · Operación Finca Potreritos
**Curso:** Uso de IA para el análisis descriptivo y predictivo en energías renovables

## La historia
La estación meteorológica de la **Finca Potreritos** (Anapoima, Cundinamarca) funciona con un **panel solar y una batería**. Si el panel no carga bien, la batería se agota y la estación se apaga. **Tu equipo es el centro de análisis** que debe mantenerla viva.

## Cómo se juega
Son **4 misiones** y cada una tiene pasos cortos. En cada paso:
1. Lee la explicación.
2. Completa la celda donde veas `...` (es lo único que debes escribir).
3. Ejecuta la celda con `Shift + Enter`. Una verificación te dirá al instante si estás bien y te dará **puntos**.
4. ¿Atascado? Abre la **Pista**. Si sigues atascado, pregunta al docente.

| Misión | De qué trata | Puntos |
|---|---|---|
| 1 | Conoce los datos de tu estación | 20 |
| 2 | Alerta de carga baja (clasificación) | 30 |
| 3 | Pronóstico del voltaje de la próxima hora | 30 |
| 4 | Cuenta tu historia (mini-presentación) | 20 |
| | **Total** | **100** |

Puedes repetir un paso las veces que quieras: se guarda tu **mejor** puntaje. Al final, el notebook te muestra un **tablero** para compartir con el docente.

## Misión 0 · Preparación
**1. Ponle nombre a tu equipo.** Escribe entre comillas, por ejemplo `'Los Voltios'`.

In [ ]:
NOMBRE_EQUIPO = ...

**2. Ejecuta la celda siguiente sin modificarla.** Carga las librerías, descarga los datos y prepara las verificaciones. No necesitas entender todo lo que hay dentro.

In [ ]:
# ---------- EJECUTAR SIN MODIFICAR ----------
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingRegressor
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_absolute_error
warnings.filterwarnings('ignore')

URL = 'https://raw.githubusercontent.com/anclopezg/curso-analisis-datos-energias-renovables/main/docs/datasets/datos_anapoima.csv'
df = pd.read_csv(URL, parse_dates=['fecha_hora'])
df = df.drop_duplicates('fecha_hora').sort_values('fecha_hora').reset_index(drop=True)   # una lectura por instante

# ----- Verificaciones y puntos -----
PUNTOS = {}
_dia = df[df['irradiancia_wm2'] > 50]
_por_hora = df.set_index('fecha_hora')['ws_panel_volt'].resample('h').mean()
_VERDAD = {'1.1': len(df), '1.2': _dia['ws_panel_volt'].mean(),
           '1.3': int(_por_hora.groupby(_por_hora.index.hour).mean().idxmax())}


def _registrar(id_, pts, maximo):
    previo = PUNTOS.get(id_, (0, maximo))[0]
    PUNTOS[id_] = (max(previo, pts), maximo)


def _vacio(r):
    return r is None or r is Ellipsis


def verificar_numero(id_, respuesta, esperado, maximo, tol, pista):
    if _vacio(respuesta):
        print('Completa la respuesta: reemplaza los ... por tu código.'); return
    try:
        ok = abs(float(respuesta) - float(esperado)) <= tol
    except (TypeError, ValueError):
        print('La respuesta debe ser un número.'); return
    if ok:
        _registrar(id_, maximo, maximo); print(f'✓ Correcto  (+{maximo} puntos)')
    else:
        print('✗ Todavía no. Pista:', pista)


def verificar_opcion(id_, respuesta, correcta, maximo, pista):
    if _vacio(respuesta):
        print("Completa la respuesta con 'a', 'b' o 'c'."); return
    if str(respuesta).strip().lower() == correcta:
        _registrar(id_, maximo, maximo); print(f'✓ Correcto  (+{maximo} puntos)')
    else:
        print('✗ Piénsalo otra vez. Pista:', pista)


def verificar_texto(id_, respuesta, maximo, minimo=25):
    if _vacio(respuesta) or not isinstance(respuesta, str) or len(respuesta.strip()) < minimo:
        print(f'Escribe al menos {minimo} caracteres con tus palabras.'); return
    _registrar(id_, maximo, maximo); print(f'✓ Registrado  (+{maximo} puntos)')


def tablero():
    filas = [(k, p, m) for k, (p, m) in sorted(PUNTOS.items())]
    total = sum(p for _, p, _ in filas)
    print(f'EQUIPO: {NOMBRE_EQUIPO}')
    for k, p, m in filas:
        print(f'  Paso {k:<5} {p:>3} / {m}')
    print(f'TOTAL: {total} / 100')
    return total


print(f'Listo. {len(df)} lecturas, del {df.fecha_hora.min():%d-%m-%Y} al {df.fecha_hora.max():%d-%m-%Y}.')
df.head(3)

---
# MISIÓN 1 · Conoce los datos de tu estación (20 puntos)
Antes de predecir nada, hay que conocer lo que se mide. La tabla `df` tiene una fila por lectura (cada 10 minutos). Las columnas que más usaremos:

| Columna | Qué es |
|---|---|
| `fecha_hora` | Cuándo se midió |
| `ws_panel_volt` | **Voltaje del panel solar** (V). Es la variable que nos importa |
| `irradiancia_wm2` | Luz solar (W/m²). Con 0 es de noche |
| `temperatura_c`, `humedad_relativa`, `presion_hpa`, `velocidad_viento_kmh` | Clima |

### Paso 1.1 · ¿Cuántas lecturas tiene `df`?
<details><summary>Pista</summary>Usa <code>len(df)</code> o <code>df.shape[0]</code>.</details>

In [ ]:
respuesta = ...
verificar_numero('1.1', respuesta, _VERDAD['1.1'], 5, 0, 'cuenta las filas de df con len(df).')

### Paso 1.2 · ¿Cuál es el voltaje promedio del panel **de día**?
"De día" significa `irradiancia_wm2 > 50`. Primero se filtran las filas de día y luego se promedia la columna del voltaje.
<details><summary>Pista</summary>Algo así: <code>df[df['irradiancia_wm2'] > 50]['ws_panel_volt'].mean()</code></details>

In [ ]:
respuesta = ...
verificar_numero('1.2', respuesta, _VERDAD['1.2'], 5, 0.05, 'filtra primero con df[df["irradiancia_wm2"] > 50] y luego usa .mean() sobre ws_panel_volt.')

### Paso 1.3 · ¿A qué hora del día el voltaje promedio es **más alto**?
Ejecuta primero esta celda: calcula el voltaje promedio de cada hora del día (0 a 23) y lo grafica.

In [ ]:
perfil = df.set_index('fecha_hora')['ws_panel_volt'].resample('h').mean()
perfil = perfil.groupby(perfil.index.hour).mean()       # promedio por hora del día
perfil.plot(kind='bar', figsize=(10, 3.5), color='tab:orange', edgecolor='black')
plt.xlabel('Hora del día'); plt.ylabel('Voltaje promedio (V)'); plt.title('Voltaje del panel por hora'); plt.show()

Mira el gráfico y responde con la hora (un número de 0 a 23).
<details><summary>Pista</summary>También puedes pedirla al código: <code>perfil.idxmax()</code> te da la hora de la barra más alta.</details>

In [ ]:
respuesta = ...
verificar_numero('1.3', respuesta, _VERDAD['1.3'], 5, 0, 'busca la barra más alta del gráfico, o usa perfil.idxmax().')

### Paso 1.4 · ¿Por qué el voltaje es casi 0 durante la noche?
- **a)** El sensor se daña por las noches.
- **b)** El panel no recibe luz del sol, así que no genera voltaje.
- **c)** La batería se agota.

Responde con la letra entre comillas, por ejemplo `'a'`.
<details><summary>Pista</summary>Mira otra vez el gráfico: ¿a qué horas sube y a qué horas baja? ¿Qué pasa con el sol a esas horas?</details>

In [ ]:
respuesta = ...
verificar_opcion('1.4', respuesta, 'b', 5, 'relaciona la forma del gráfico con las horas de sol.')

---
# MISIÓN 2 · Alerta de carga baja (30 puntos)
**Objetivo:** que la estación **avise** cuando el panel está cargando mal. Usaremos solo las horas de día (de noche el voltaje es 0 y no hay nada que avisar).

Vamos a **clasificar** cada lectura de día en dos categorías:
- **`alerta = 1`** → carga baja (voltaje por debajo de un umbral)
- **`alerta = 0`** → todo bien

### Paso 2.1 · Crear la alerta
Una lectura es alerta si el voltaje es **menor que 12.5 V**. Reemplaza `...` por ese número.
<details><summary>Pista</summary>Es el número 12.5.</details>

In [ ]:
dia = df[df['irradiancia_wm2'] > 50].dropna(subset=['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa', 'ws_panel_volt']).copy()

UMBRAL_ALERTA = ...

if _vacio(UMBRAL_ALERTA):
    print('Completa UMBRAL_ALERTA: reemplaza los ... por el número.')
else:
    dia['alerta'] = (dia['ws_panel_volt'] < UMBRAL_ALERTA).astype(int)
    verificar_numero('2.1', dia['alerta'].mean(), 0.353, 5, 0.005, 'el umbral debe ser 12.5 V.')

### Paso 2.2 · Separar el pasado del futuro
Entrenamos con lo que pasó **antes del 1 de septiembre de 2026** y probamos con lo que pasó **después** (como en la vida real: el modelo debe predecir lo que aún no ha visto). Ejecuta esta celda sin modificarla.

In [ ]:
assert 'alerta' in dia.columns, 'Completa antes el paso 2.1 (UMBRAL_ALERTA).'
VARIABLES = ['temperatura_c', 'irradiancia_wm2', 'velocidad_viento_kmh', 'humedad_relativa', 'presion_hpa']
corte = pd.Timestamp('2026-09-01')
entrenamiento, prueba = dia[dia['fecha_hora'] < corte], dia[dia['fecha_hora'] >= corte]
X_train, y_train = entrenamiento[VARIABLES], entrenamiento['alerta']
X_test, y_test = prueba[VARIABLES], prueba['alerta']
print(f'Entrenamiento: {len(X_train)} lecturas | Prueba: {len(X_test)} lecturas')


def entrenar_clasificador(n_arboles, profundidad):
    if _vacio(n_arboles) or not isinstance(n_arboles, (int, np.integer)) or not 20 <= n_arboles <= 500:
        print('✗ n_arboles debe ser un número entero entre 20 y 500.'); return None
    if not (profundidad is None or (isinstance(profundidad, (int, np.integer)) and 2 <= profundidad <= 30)):
        print('✗ profundidad debe ser un entero entre 2 y 30, o None (sin límite).'); return None
    m = RandomForestClassifier(n_estimators=n_arboles, max_depth=profundidad, min_samples_leaf=5, random_state=0, n_jobs=-1)
    m.fit(X_train, y_train)
    _registrar('2.3', 5, 5)
    print(f'✓ Modelo entrenado con {n_arboles} árboles y profundidad {profundidad}  (+5 puntos)')
    return m

**¿Qué porcentaje de las lecturas de entrenamiento son alerta?** Es importante saberlo: si casi nunca hay alerta, el modelo podría "acertar" diciendo siempre que todo está bien.

### Paso 2.2b · Calcula ese porcentaje (un número de 0 a 100)
<details><summary>Pista</summary><code>y_train</code> vale 1 o 0. Su promedio es la fracción de alertas; multiplícalo por 100.</details>

In [ ]:
prevalencia = ...
verificar_numero('2.2', prevalencia, 36.4, 5, 0.5, 'y_train.mean() * 100.')

### Paso 2.3 · Elegir y entrenar tu modelo
Usaremos un **Random Forest**: muchos árboles de decisión que "votan". Tú decides dos cosas:
- **`n_arboles`:** cuántos árboles (entre 20 y 500). Más árboles = resultado más estable, pero más lento.
- **`profundidad`:** qué tan profundo puede ser cada árbol (entre 2 y 30, o `None` sin límite). Poco profundo = modelo simple que puede quedarse corto; muy profundo = puede memorizar los datos.

No hay una respuesta única: **prueba valores** y mira cómo cambian los resultados del siguiente paso. Un buen punto de partida: 100 árboles y profundidad 10.

In [ ]:
n_arboles = ...
profundidad = ...

modelo = entrenar_clasificador(n_arboles, profundidad)

### Paso 2.4 · Ajustar la sensibilidad de la alerta
El modelo calcula una **probabilidad** de alerta para cada lectura. Tú decides desde qué probabilidad se activa el aviso (`umbral_decision`):
- **Umbral alto (0.7):** avisa solo cuando está muy seguro → pocas falsas alarmas, pero se le **escapan** alertas reales.
- **Umbral bajo (0.2):** avisa ante la mínima duda → detecta casi todas las alertas, pero **molesta con muchas falsas alarmas**.

Las dos métricas que importan:
- **Recall:** de las alertas reales, ¿cuántas detectó? (que no se escape ninguna)
- **Precisión:** de los avisos que dio, ¿cuántos eran reales? (que no moleste de más)

**Tu meta:** recall de **al menos 90 %** y precisión de **al menos 75 %**. (Con solo cumplir el recall ganas 5 puntos; cumpliendo ambas, 10.)

Cambia `umbral_decision` (un número entre 0 y 1) y vuelve a ejecutar hasta lograrlo.
<details><summary>Pista</summary>Si el recall es bajo, baja el umbral. Si la precisión es baja, súbelo. Si no logras las dos, prueba cambiar el modelo del paso 2.3 (más árboles o distinta profundidad).</details>

In [ ]:
umbral_decision = 0.35      # <-- cámbialo

assert modelo is not None, 'Primero completa el paso 2.3'
probabilidad = modelo.predict_proba(X_test)[:, 1]
prediccion = (probabilidad >= umbral_decision).astype(int)

recall = recall_score(y_test, prediccion)
precision = precision_score(y_test, prediccion)
print(f'Recall    : {recall:.1%}   (meta: 90 % o más)')
print(f'Precisión : {precision:.1%}   (meta: 75 % o más)')
if recall >= 0.90 and precision >= 0.75:
    _registrar('2.4', 10, 10); print('✓ Meta cumplida  (+10 puntos)')
elif recall >= 0.90:
    _registrar('2.4', 5, 10); print('Casi: detectas las alertas, pero con demasiadas falsas alarmas  (+5 puntos). Intenta mejorar la precisión.')
else:
    print('✗ Aún se te escapan alertas. Baja el umbral o mejora el modelo.')

**Para ver el equilibrio:** esta celda muestra qué pasaría con distintos umbrales (ejecútala sin modificar).

In [ ]:
filas = []
for u in [0.2, 0.3, 0.4, 0.5, 0.6, 0.7]:
    p = (probabilidad >= u).astype(int)
    filas.append({'umbral': u, 'recall': round(recall_score(y_test, p), 3), 'precisión': round(precision_score(y_test, p), 3)})
pd.DataFrame(filas).set_index('umbral')

### Paso 2.5 · Pregunta de reflexión
En esta estación, ¿cuál de estos dos errores es **más grave**?
- **a)** El modelo avisa "carga baja" pero en realidad estaba bien (falsa alarma).
- **b)** El modelo **no avisa** cuando la carga sí estaba baja (se le escapó una alerta).
- **c)** Los dos son igual de graves.
<details><summary>Pista</summary>Piensa qué pasa con la batería y la estación si nadie se entera de que el panel carga mal.</details>

In [ ]:
respuesta = ...
verificar_opcion('2.5', respuesta, 'b', 5, 'una alerta que no se detecta puede dejar la estación sin energía.')

---
# MISIÓN 3 · Pronóstico del voltaje de la próxima hora (30 puntos)
**Objetivo:** predecir cuántos **voltios** tendrá el panel dentro de una hora (un número, no una categoría). Así se puede decidir con anticipación si hay que apagar sensores.

Para predecir el futuro solo podemos usar el **pasado**. La tabla que se crea abajo tiene, para cada hora, lo que pasó antes. Los nombres terminan en `_lag` + el número de horas atrás:

| Columna | Significado |
|---|---|
| `voltaje` | **Lo que queremos predecir** (voltaje de esa hora) |
| `volt_lag1`, `volt_lag2`, `volt_lag3` | Voltaje de hace 1, 2 y 3 horas |
| `volt_lag24` | Voltaje de hace 24 horas, es decir **ayer a la misma hora** |
| `irr_lag1`, `irr_lag24` | Luz solar de hace 1 y 24 horas |
| `temp_lag1` | Temperatura de hace 1 hora |
| `hora_sin`, `hora_cos` | La hora del día (codificada para que las 23 h y las 0 h queden cerca) |

Ejecuta esta celda sin modificarla.

In [ ]:
cols = ['ws_panel_volt', 'temperatura_c', 'irradiancia_wm2']
h = df.set_index('fecha_hora')[cols].resample('h').mean().interpolate(limit=3, limit_area='inside')
tabla = pd.DataFrame({
    'voltaje': h['ws_panel_volt'],
    'volt_lag1': h['ws_panel_volt'].shift(1), 'volt_lag2': h['ws_panel_volt'].shift(2), 'volt_lag3': h['ws_panel_volt'].shift(3),
    'volt_lag24': h['ws_panel_volt'].shift(24),
    'irr_lag1': h['irradiancia_wm2'].shift(1), 'irr_lag24': h['irradiancia_wm2'].shift(24),
    'temp_lag1': h['temperatura_c'].shift(1),
    'hora_sin': np.sin(2 * np.pi * h.index.hour / 24), 'hora_cos': np.cos(2 * np.pi * h.index.hour / 24)}).dropna()

ent3, prue3 = tabla[tabla.index < corte], tabla[tabla.index >= corte]       # mismo corte que en la misión 2
DISPONIBLES = [col for col in tabla.columns if col != 'voltaje']
print(f'Entrenamiento: {len(ent3)} horas | Prueba: {len(prue3)} horas')
print('Columnas que puedes usar:', DISPONIBLES)

### Paso 3.1 · La vara para medir: "ayer a la misma hora"
Un modelo solo vale la pena si le **gana a una regla sencilla**. La regla: *"el voltaje de ahora será igual al de hace 24 horas"*.

**3.1a)** ¿Cuál columna de la tabla es "ayer a la misma hora"? Escribe su nombre entre comillas.
<details><summary>Pista</summary>Es la que termina en 24.</details>

In [ ]:
columna_ayer = ...
verificar_opcion('3.1', columna_ayer, 'volt_lag24', 5, 'la columna del voltaje de hace 24 horas.')

**3.1b)** Con esa columna se mide cuánto se equivoca la regla. Ejecuta la celda: el resultado es el **error promedio (MAE) en voltios** que tu modelo debe superar.

In [ ]:
assert not _vacio(columna_ayer) and columna_ayer in tabla.columns, 'Completa antes el paso 3.1a con el nombre de la columna.'
mae_ayer = mean_absolute_error(prue3['voltaje'], prue3[columna_ayer])
print(f'La regla "ayer a la misma hora" se equivoca en promedio {mae_ayer:.3f} V.  Tu modelo debe lograr menos que eso.')

### Paso 3.2 · Elegir las variables de tu modelo
Escribe una **lista** con los nombres de las columnas que usará tu modelo (entre comillas, separadas por comas). Puedes usar cualquiera de las que aparecieron en `DISPONIBLES`.
- Usar **pocas** variables = modelo simple, pero puede perder información.
- Usar **muchas** puede ayudar, pero no siempre.

Ejemplo de formato: `variables = ['volt_lag1', 'volt_lag24']`
<details><summary>Pista</summary>Piensa qué ayuda a predecir el voltaje de la próxima hora: el voltaje reciente (<code>volt_lag1</code>...), el de ayer, la luz, la hora del día... Un modelo que incluye la <b>hora del día</b> (<code>hora_sin</code> y <code>hora_cos</code>) suele ver mejor los saltos del amanecer y del atardecer.</details>

In [ ]:
variables = [...]

if _vacio(variables) or variables == [Ellipsis] or not isinstance(variables, list):
    print('Escribe tu lista de variables.')
elif 'voltaje' in variables or any(v not in DISPONIBLES for v in variables):
    print('✗ Usa solo nombres de la lista DISPONIBLES (y no incluyas "voltaje": esa es la respuesta).')
elif len(variables) < 2:
    print('Elige al menos 2 variables.')
else:
    _registrar('3.2', 5, 5); print(f'✓ {len(variables)} variables elegidas  (+5 puntos)')

### Paso 3.3 · Entrenar y comparar
Esta celda entrena un modelo con **tus variables** y lo compara con la regla de "ayer". Ejecútala sin modificarla.
**Puntos:** 10 si le ganas a la regla y 10 más si la mejoras en **25 % o más**.

Si no llegas, vuelve al paso 3.2, cambia tus variables y ejecuta de nuevo.

In [ ]:
assert isinstance(variables, list) and variables and Ellipsis not in variables and all(v in DISPONIBLES for v in variables), 'Completa antes el paso 3.2 con una lista válida de variables.'
modelo3 = HistGradientBoostingRegressor(max_iter=200, learning_rate=0.05, random_state=0)
modelo3.fit(ent3[variables], ent3['voltaje'])
pred3 = modelo3.predict(prue3[variables])

mae_modelo = mean_absolute_error(prue3['voltaje'], pred3)
mejora = 1 - mae_modelo / mae_ayer
print(f'Regla "ayer a la misma hora": {mae_ayer:.3f} V de error')
print(f'Tu modelo                   : {mae_modelo:.3f} V de error   ->  mejora de {mejora:.1%}')
if mejora > 0:
    _registrar('3.3a', 10, 10); print('✓ Le ganaste a la regla de ayer  (+10 puntos)')
else:
    print('✗ Todavía no superas a la regla de ayer. Prueba con otras variables.')
if mejora >= 0.25:
    _registrar('3.3b', 10, 10); print('✓ Mejora de 25 % o más  (+10 puntos)')
else:
    print('Para el segundo bono necesitas una mejora de 25 % o más.')

Mira cómo le fue a tu modelo durante 4 días (las primeras 96 horas de prueba). Ejecuta sin modificar.

In [ ]:
n = 96
plt.figure(figsize=(12, 3.8))
plt.plot(prue3.index[:n], prue3['voltaje'].values[:n], 'k', lw=2, label='real')
plt.plot(prue3.index[:n], prue3[columna_ayer].values[:n], alpha=0.6, label='regla: ayer misma hora')
plt.plot(prue3.index[:n], pred3[:n], label='tu modelo')
plt.ylabel('Voltaje (V)'); plt.title('Real contra predicho'); plt.legend(ncol=3); plt.grid(alpha=0.3); plt.show()

---
# MISIÓN 4 · Cuenta tu historia (20 puntos)
Un buen análisis no termina con un número: termina con una **decisión** y con **honestidad**. Completa estos tres pasos y luego ejecuta la celda final para generar tu ficha de presentación (3 minutos).

### Paso 4.1 · ¿Qué decisión apoya tu trabajo?
- **a)** Apagar sensores no críticos cuando el modelo anticipe voltaje bajo o lance una alerta, para proteger la batería.
- **b)** Comprar de inmediato un panel solar más grande.
- **c)** Trasladar la estación a otro lugar.
<details><summary>Pista</summary>¿Qué puedes decidir con lo que mostraron tus modelos, sin información adicional?</details>

In [ ]:
decision = ...
verificar_opcion('4.1', decision, 'a', 5, 'elige lo que tus resultados realmente permiten decidir.')

### Paso 4.2 · Una limitación de tu trabajo
Sé honesto: ¿en qué puede fallar tu modelo? Escribe una frase con tus palabras.
<details><summary>Pista</summary>Ideas: ¿qué pasa en el amanecer o el atardecer? ¿Y si cambia la época del año? ¿Qué pasó con los días en que la estación no registró datos? ¿Cuántas falsas alarmas da tu alerta?</details>

In [ ]:
limitacion = ...
verificar_texto('4.2', limitacion, 5)

### Paso 4.3 · Qué harían con más tiempo
Escribe una frase con el siguiente paso que darían.
<details><summary>Pista</summary>Ideas: agregar nubosidad o pronóstico del clima, probar con otra estación, probar el modelo en vivo.</details>

In [ ]:
proximo_paso = ...
verificar_texto('4.3', proximo_paso, 5)

### Paso 4.4 · Tu ficha y tu tablero
Ejecuta esta celda para generar tu ficha de presentación y ver tu puntaje. **Compártela con el docente**.

In [ ]:
opciones = {'a': 'apagar sensores no críticos cuando se anticipe voltaje bajo, para proteger la batería',
            'b': 'comprar un panel más grande', 'c': 'trasladar la estación'}
if NOMBRE_EQUIPO is Ellipsis:
    print('Vuelve a la Misión 0 y escribe el nombre de tu equipo.')
else:
    _registrar('4.4', 5, 5)
    print(f'''
=========== FICHA DEL EQUIPO "{NOMBRE_EQUIPO}" ===========
DECISIÓN: {opciones.get(str(decision).lower(), decision)}.

RESULTADOS
  - Alerta de carga baja: recall {recall:.0%} y precisión {precision:.0%} (umbral de decisión {umbral_decision}).
  - Pronóstico a una hora: error de {mae_modelo:.2f} V frente a {mae_ayer:.2f} V de la regla "ayer a la misma hora" (mejora de {mejora:.0%}).

LIMITACIÓN: {limitacion}
PRÓXIMO PASO: {proximo_paso}
===========================================================
''')
    tablero()

---
## ¡Listo!
Compartan en voz alta: **nombre del equipo, puntaje total, y su resultado en recall y en mejora sobre "ayer"**. Gana el equipo con más puntos; en caso de empate, el que tenga la mejor explicación de su limitación.

**Lo que aplicaste del curso:** explorar datos (sesiones 2 y 4), clasificar con Random Forest y ajustar el umbral (6), predecir con variables rezagadas y compararse contra una línea base (7, 8 y 9), y comunicar resultados con honestidad (10 y 11).